# Spatial Analysis of Vacant Dwellings per Inhabitant in Catalonia (2025)

* **Author:** Adrià Benito Carayol
* **Project:** `Speculation-Map` — Big Data Analysis and Visualization (University of Barcelona)

### Notebook Objective
This notebook analyzes the territorial distribution of vacant dwellings (*Inventari d'Habitatges Buits* — IHB) normalized by population across Catalonia for the year **2025**. Using a logarithmic color scale to handle strong territorial disparities, it examines:
1. **Comarca-level distribution:** Aggregated ratio of vacant dwellings per inhabitant across Catalan *comarques*.
2. **Municipal-level distribution:** High-resolution spatial breakdown of vacant dwellings per inhabitant across individual municipalities.

### Data Sources
* **Administrative boundaries:** Cartographic and Geological Institute of Catalonia (`CatalunyaMapComarques.json` and `CatalunyaMapMunicipalities.json`, projected to ETRS89 / UTM zone 31N — `EPSG:25831`).
* **Vacant dwellings (2025):** *Inventari d'Habitatges Buits* (IHB) dataset (`EmptyBuildings.csv`).
* **Municipal population (2025):** Statistical Institute of Catalonia (Idescat, `Habitants.csv`).

### AI Usage:

AI tools were used as a support in some parts of the creation of the figures (for example, to add scale bars to the maps).

---

## 1. Vacant Dwellings per Inhabitant by Comarca (2025)
This map aggregates municipal population and vacant dwelling counts at the *comarca* level (`CODICOMAR`) to visualize regional disparities in housing vacancy relative to resident population on a logarithmic scale.

In [ ]:
# Vacant dwellings (IHB) per inhabitant by comarca in Catalonia (2025 - logarithmic scale)

import geopandas as gpd
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

# municipis and comarcas
municipis = gpd.read_file("../Data/CatalunyaMapMunicipalities.json")
municipis["CODIMUNI"] = municipis["CODIMUNI"].astype(str).str.zfill(6)
municipis["CODICOMAR"] = municipis["CODICOMAR"].astype(str).str.zfill(2)
mapa_comarques = municipis[["CODIMUNI", "CODICOMAR"]].drop_duplicates()

comarques = gpd.read_file("../Data/CatalunyaMapComarques.json")
comarques["CODICOMAR"] = comarques["CODICOMAR"].astype(str).str.zfill(2)
comarques = comarques.to_crs(epsg=25831)

# 2. 2025 population
df_pob = pd.read_csv("../Data/Habitants.csv", dtype=str)
pob_2025 = df_pob[
    (df_pob["YEAR"] == "2025")
    & (df_pob["SEX"] == "TOTAL")
    & (df_pob["CONCEPT"] == "POP")
].copy()
pob_2025["Codi"] = pob_2025["MUN"].str.strip().str.zfill(6)
pob_2025 = pob_2025[pob_2025["Codi"].str.match(r"^\d{6}$")].copy()
pob_2025["Poblacio_2025"] = pd.to_numeric(pob_2025["value"], errors="coerce")

pob_comarca = (
    pob_2025.merge(mapa_comarques, left_on="Codi", right_on="CODIMUNI", how="inner")
    .groupby("CODICOMAR", as_index=False)["Poblacio_2025"]
    .sum()
)

# 3. vacant dwellings 2025
df_ihb = pd.read_csv("../Data/EmptyBuildings/EmptyBuildings.csv", dtype=str)
df_ihb_2025 = df_ihb[df_ihb["Any"] == "2025"].copy()
df_ihb_2025["Codi"] = df_ihb_2025["Codi"].str.strip().str.zfill(6)
df_ihb_2025["Habitatges"] = pd.to_numeric(
    df_ihb_2025["Habitatges"].str.replace(".", "", regex=False),
    errors="coerce",
)

buits_comarca = (
    df_ihb_2025.merge(mapa_comarques, left_on="Codi", right_on="CODIMUNI", how="inner")
    .groupby("CODICOMAR", as_index=False)["Habitatges"]
    .sum()
    .rename(columns={"Habitatges": "Num_buits"})
)


df_comarca = pob_comarca.merge(buits_comarca, on="CODICOMAR", how="left")
df_comarca = df_comarca[(df_comarca["Poblacio_2025"] > 0) & (df_comarca["Num_buits"] > 0)].copy()
df_comarca["Buits_per_hab"] = df_comarca["Num_buits"] / df_comarca["Poblacio_2025"]

gdf_comarques = comarques.merge(df_comarca[["CODICOMAR", "Buits_per_hab"]], on="CODICOMAR", how="left")

fig, ax = plt.subplots(figsize=(8.5, 8), layout="constrained")

norm = colors.LogNorm(
    vmin=gdf_comarques["Buits_per_hab"].min(),
    vmax=gdf_comarques["Buits_per_hab"].max(),
)

gdf_comarques.plot(
    column="Buits_per_hab",
    cmap="Purples",
    norm=norm,
    edgecolor="dimgrey",
    linewidth=0.5,
    legend=True,
    legend_kwds={
        "label": "Vacant dwellings per inhabitant (logarithmic scale)",
        "shrink": 0.7,
        "format": ticker.FuncFormatter(lambda x, _: f"{x:g}"),
    },
    missing_kwds={"color": "#e0e0e0", "hatch": "///", "label": "No data / 0"},
    ax=ax,
)

# North arrow
ax.annotate(
    "N",
    xy=(0.08, 0.94),
    xytext=(0.08, 0.85),
    xycoords="axes fraction",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    arrowprops=dict(arrowstyle="-|>", facecolor="black", edgecolor="black", lw=1.8, mutation_scale=18),
)

# scale bar
xmin, ymin, xmax, ymax = gdf_comarques.total_bounds
longitud_total = 50_000
tram = longitud_total / 2
alcada_barra = (ymax - ymin) * 0.012
x_inici = xmax - longitud_total - (xmax - xmin) * 0.04
y_inici = ymin + (ymax - ymin) * 0.04

ax.add_patch(mpatches.Rectangle((x_inici, y_inici), tram, alcada_barra, facecolor="black", edgecolor="black", lw=0.8))
ax.add_patch(mpatches.Rectangle((x_inici + tram, y_inici), tram, alcada_barra, facecolor="white", edgecolor="black", lw=0.8))
for offset, etiqueta in [(0, "0"), (tram, "25"), (longitud_total, "50 km")]:
    ax.text(x_inici + offset, y_inici - alcada_barra * 0.6, etiqueta, ha="center", va="top", fontsize=8)

ax.set_title("Vacant Dwellings per Inhabitant by Comarca in Catalonia (2025)", fontweight="bold")
ax.set_axis_off()
plt.show()

## 2. Vacant Dwellings per Inhabitant by Municipality (2025)
This map increases spatial granularity to the municipal level (`CODIMUNI`), revealing localized concentrations of vacant dwellings per inhabitant across Catalonia in 2025 (with hatched grey areas indicating municipalities without IHB records or zero reported vacant units).

In [ ]:
# Vacant dwellings (IHB) per inhabitant by municipality in Catalonia (2025 - logarithmic scale)

import geopandas as gpd
import matplotlib.colors as colors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

municipis = gpd.read_file("../Data/CatalunyaMapMunicipalities.json")
municipis["CODIMUNI"] = municipis["CODIMUNI"].astype(str).str.zfill(6)
municipis = municipis.to_crs(epsg=25831)

# 2. 2025 POPULATION 
df_pob = pd.read_csv("../Data/Habitants.csv", dtype=str)
pob_2025 = df_pob[
    (df_pob["YEAR"] == "2025")
    & (df_pob["SEX"] == "TOTAL")
    & (df_pob["CONCEPT"] == "POP")
].copy()
pob_2025["Codi"] = pob_2025["MUN"].str.strip().str.zfill(6)
pob_2025 = pob_2025[pob_2025["Codi"].str.match(r"^\d{6}$")].copy()
pob_2025["Poblacio_2025"] = pd.to_numeric(pob_2025["value"], errors="coerce")
pob_2025 = pob_2025[["Codi", "Poblacio_2025"]].dropna()

# 3. 2025 VACANT DWELLINGS 
df_ihb = pd.read_csv("../Data/EmptyBuildings/EmptyBuildings.csv", dtype=str)
df_ihb_2025 = df_ihb[df_ihb["Any"] == "2025"].copy()
df_ihb_2025["Codi"] = df_ihb_2025["Codi"].str.strip().str.zfill(6)
df_ihb_2025["Habitatges"] = pd.to_numeric(
    df_ihb_2025["Habitatges"].str.replace(".", "", regex=False),
    errors="coerce",
)
buits_per_muni = (
    df_ihb_2025.groupby("Codi", as_index=False)["Habitatges"]
    .sum()
    .rename(columns={"Habitatges": "Num_buits"})
)

df_combinat = pob_2025.merge(buits_per_muni, on="Codi", how="inner")
df_combinat = df_combinat[(df_combinat["Poblacio_2025"] > 0) & (df_combinat["Num_buits"] > 0)].copy()
df_combinat["Buits_per_hab"] = df_combinat["Num_buits"] / df_combinat["Poblacio_2025"]

gdf_municipis = municipis.merge(df_combinat[["Codi", "Buits_per_hab"]], left_on="CODIMUNI", right_on="Codi", how="left")

fig, ax = plt.subplots(figsize=(8.5, 8), layout="constrained")

norm = colors.LogNorm(
    vmin=gdf_municipis["Buits_per_hab"].min(),
    vmax=gdf_municipis["Buits_per_hab"].max(),
)

gdf_municipis.plot(
    column="Buits_per_hab",
    cmap="Purples",
    norm=norm,
    edgecolor="lightgrey",
    linewidth=0.2,
    legend=True,
    legend_kwds={
        "label": "Vacant dwellings per inhabitant (logarithmic scale)",
        "shrink": 0.7,
        "format": ticker.FuncFormatter(lambda x, _: f"{x:g}"),
    },
    missing_kwds={"color": "#e0e0e0", "hatch": "///", "label": "No IHB record / 0"},
    ax=ax,
)

# North arrow
ax.annotate(
    "N",
    xy=(0.08, 0.94),
    xytext=(0.08, 0.85),
    xycoords="axes fraction",
    ha="center",
    va="center",
    fontsize=12,
    fontweight="bold",
    arrowprops=dict(arrowstyle="-|>", facecolor="black", edgecolor="black", lw=1.8, mutation_scale=18),
)

# scale bar
xmin, ymin, xmax, ymax = gdf_municipis.total_bounds
longitud_total = 50_000
tram = longitud_total / 2
alcada_barra = (ymax - ymin) * 0.012
x_inici = xmax - longitud_total - (xmax - xmin) * 0.04
y_inici = ymin + (ymax - ymin) * 0.04

ax.add_patch(mpatches.Rectangle((x_inici, y_inici), tram, alcada_barra, facecolor="black", edgecolor="black", lw=0.8))
ax.add_patch(mpatches.Rectangle((x_inici + tram, y_inici), tram, alcada_barra, facecolor="white", edgecolor="black", lw=0.8))
for offset, etiqueta in [(0, "0"), (tram, "25"), (longitud_total, "50 km")]:
    ax.text(x_inici + offset, y_inici - alcada_barra * 0.6, etiqueta, ha="center", va="top", fontsize=8)

ax.set_title("Vacant Dwellings per Inhabitant by Municipality in Catalonia (2025)", fontweight="bold")
ax.set_axis_off()
plt.show()